# Evaluation layer

Task 4. One evaluation path for M0-M4 and D0: predictions -> metrics -> deltas ->
results. All logic lives in `src/evaluation.py`, `src/results.py`,
`src/checkpoint.py` - this notebook just shows it working. Full coverage is in
`tests/test_evaluation.py` and `tests/test_results.py` (35 tests).

## Check metrics

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
from sklearn.metrics import f1_score

from src.data import LABELS
from src.evaluation import compute_metrics, compute_val_macro_f1

rng = np.random.default_rng(42)
y_true = rng.integers(0, len(LABELS), size=300)
y_pred = y_true.copy()
flip = rng.random(300) < 0.25
y_pred[flip] = rng.integers(0, len(LABELS), size=int(flip.sum()))

res = compute_metrics(y_true, y_pred, labels=LABELS)
print("macro_f1:", res["macro_f1"])
print("sklearn :", f1_score(y_true, y_pred, average="macro", zero_division=0))
print("accuracy, macro_precision, macro_recall:",
      res["accuracy"], res["macro_precision"], res["macro_recall"])

macro_f1: 0.766167009504296
sklearn : 0.766167009504296
accuracy, macro_precision, macro_recall: 0.7666666666666667 0.7678239616987798 0.7666236455436389


In [2]:
import pandas as pd

pd.DataFrame(res["per_class"]).T

,precision,recall,f1,support
Checking or savings account,0.774194,0.813559,0.793388,59.0
Credit card,0.744681,0.729167,0.736842,48.0
Debt collection,0.731707,0.789474,0.759494,76.0
"Money transfer, virtual currency, or money service",0.822581,0.718310,0.766917,71.0
Student loan,0.765957,0.782609,0.774194,46.0


In [3]:
pd.DataFrame(res["confusion_matrix"], index=LABELS, columns=LABELS)

,Checking or savings account,Credit card,Debt collection,"Money transfer, virtual currency, or money service",Student loan
Checking or savings account,48,3,6,1,1
Credit card,0,35,2,5,6
Debt collection,7,4,60,3,2
"Money transfer, virtual currency, or money service",3,4,11,51,2
Student loan,4,1,3,2,36


`compute_metrics` fails loudly rather than silently on bad input - mismatched
lengths, empty arrays, or a label id outside `[0, 5)` all raise `ValueError`
instead of quietly dropping a class from the average.

In [4]:
try:
    compute_metrics([0, 1, 5], [0, 1, 2], labels=LABELS)
except ValueError as e:
    print("raised as expected:", e)

raised as expected: Label id(s) [5] fall outside the valid range [0, 5) for 5 canonical classes


`val_macro_f1` - the checkpoint-selection metric - is `compute_val_macro_f1`,
computed once per epoch on the full validation set. It must not be a running
per-batch average; macro-F1 doesn't decompose across batches. A quick
demonstration:

In [5]:
correct = compute_val_macro_f1(y_true, y_pred, labels=LABELS)

# Naive (wrong) pattern: score each 10-row batch, then average the scores.
batch_scores = [
    compute_val_macro_f1(y_true[i:i+10], y_pred[i:i+10], labels=LABELS)
    for i in range(0, len(y_true), 10)
]
naive = float(np.mean(batch_scores))

print("full-set val_macro_f1     :", correct)
print("naive per-batch average   :", naive)
print("difference                :", abs(correct - naive))

full-set val_macro_f1     : 0.766167009504296
naive per-batch average   : 0.7198729141229143
difference                : 0.04629409538138163


## Check deltas

In [6]:
from src.evaluation import calculate_deltas, format_delta

m0, m1, m2 = 0.8420, 0.8580, 0.8530  # M2 regresses slightly vs M1, still above M0

for name, current, prev in [("M0", m0, None), ("M1", m1, m0), ("M2", m2, m1)]:
    d = calculate_deltas(current_f1=current, previous_f1=prev, baseline_m0_f1=m0 if name != "M0" else None)
    print(f"{name}: vs previous {format_delta(d['delta_vs_previous'])}  |  vs M0 {format_delta(d['delta_vs_m0'])}")

M0: vs previous —  |  vs M0 —
M1: vs previous +0.0160  |  vs M0 +0.0160
M2: vs previous -0.0050  |  vs M0 +0.0110


Incremental delta (`delta_vs_previous`) is what this rung contributed on its own;
cumulative delta (`delta_vs_m0`) is the total improvement over the baseline. Both
are computed on unrounded floats - rounding only happens in `format_delta` for
display.

## Check mean/std across seeds

In [7]:
from src.evaluation import seed_statistics

m0_seed_f1s = [0.8410, 0.8430, 0.8420]
stats = seed_statistics(m0_seed_f1s)
print(stats)  # ddof=1 sample std - 3 seeds is a sample, not the whole population

single = seed_statistics([0.8580])  # M1 is single-seed
print(single)  # std is None, not 0 - no spread was actually measured

{'mean': 0.842, 'std': 0.0010000000000000009, 'n': 3, 'ddof': 1}
{'mean': 0.858, 'std': None, 'n': 1, 'ddof': 1}


## Validate results

In [8]:
import shutil
import tempfile
from pathlib import Path

from src.results import log_run_to_csv, generate_comparison_table, validate_runs_csv

tmp = Path(tempfile.mkdtemp())
runs_csv = tmp / "runs.csv"

base = {
    "model": "demo", "best_epoch": 5, "epochs_run": 10,
    "train_time": 100.0, "parameter_count": 150000,
    "checkpoint_path": "checkpoints/demo.weights.h5",
    "dataset_version": "demo",
}
for seed, f1 in [(42, 0.8410), (123, 0.8430), (456, 0.8420)]:
    log_run_to_csv({**base, "experiment": "M0", "seed": seed,
                     "macro_f1": f1, "accuracy": f1 + 0.003,
                     "macro_precision": f1, "macro_recall": f1}, runs_csv_path=runs_csv)
log_run_to_csv({**base, "experiment": "M1", "seed": 42,
                 "macro_f1": 0.8580, "accuracy": 0.8610,
                 "macro_precision": 0.858, "macro_recall": 0.858}, runs_csv_path=runs_csv)

is_valid, errors = validate_runs_csv(runs_csv)
print("valid:", is_valid, "| errors:", errors)
generate_comparison_table(runs_csv_path=runs_csv)

valid: True | errors: []


,Model,Configuration,Macro-F1,Accuracy,Δ vs Previous,Δ vs M0,Interpretation
0,M0,"Unidirectional LSTM baseline, random embeddings",0.8420 ± 0.0010,0.8450 ± 0.0010,—,—,
1,M1,Bidirectional LSTM,0.8580,0.8610,+0.0160,+0.0160,


In [9]:
shutil.rmtree(tmp, ignore_errors=True)

`generate_comparison_table` reads only from `results/runs.csv` - never hand-edited
- and reuses `seed_statistics`/`calculate_deltas` rather than recomputing mean/std
or deltas inline, so the comparison table and any other consumer of these numbers
can't drift apart.